# 1. Penggabungan Data dan Perbandingan Populasi

Notebook pertama dari empat notebook **Gabungan**. Dua pekerjaan sekaligus:

1. **Menyusun dataset gabungan** dari hasil bersih Bantul dan Sleman, dengan satu
   kolom baru `wilayah`. Dataset inilah yang dipakai notebook `3. Model Gabungan`.
2. **Membandingkan kedua populasi** apa adanya: berapa banyak pesertanya, seperti
   apa sebaran usia dan jenis kelaminnya, segmen kepesertaannya, status gizinya,
   dan berapa prevalensi tiap penyakit.

### Tiga aturan pembacaan yang berlaku di seluruh notebook ini

**Setiap angka prevalensi selalu disertai pita usianya.** Label
`Tidak Dapat Pertanyaan` adalah **gerbang kelayakan kuesioner**, bukan data
hilang. Diabetes dan Hipertensi hanya ditanyakan pada usia 15+, Kanker Paru pada
usia 45+, TBC pada semua usia. Jadi keempat penyakit punya **subpopulasi yang
berbeda**, dan membandingkan prevalensinya tanpa menyebut pita usia adalah
membandingkan dua hal yang berbeda.

**Yang digambar adalah prevalensi, bukan jumlah orang berisiko.** Jumlah orang
berisiko pada suatu kelompok sama dengan `jumlah peserta x risiko`, dan suku
pertama biasanya mendominasi. Grafik jumlah bisa membalik kesimpulan.

**Semua angka berlaku per catatan skrining, bukan per orang.** Kolom `id` sengaja
diacak oleh BPJS untuk melindungi data peserta asli, sehingga tidak ada cara
mengenali orang yang sama dua kali. Setelah digabung, `id` menjadi semakin tidak
bermakna karena kedua wilayah memakai rentang nilai yang sama.

In [ ]:
# ===================== IMPORTS =====================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Akar proyek dicari dari folder kerja -- tidak ada jalur absolut di notebook
# ini, jadi seluruh folder boleh dipindah.
AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg
import bmi_standards as bs
import viz_style as vz

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 190)

WILAYAH = ["Bantul", "Sleman"]
WARNA = vz.PALETTE["WILAYAH"]

# Nama pendek -> kolom target, dipakai di seluruh notebook Gabungan.
TARGETS = cfg.DISEASE_TARGETS
NAMA_PENYAKIT = {
    "diabetes": "Diabetes Melitus",
    "hipertensi": "Hipertensi",
    "kanker_paru": "Kanker Paru",
    "tbc": "TBC",
}

# Pita usia tempat tiap kuesioner ditanyakan. WAJIB ikut disebut di setiap
# angka prevalensi: "Tidak Dapat Pertanyaan" adalah gerbang kelayakan, bukan
# data hilang, sehingga tiap penyakit punya subpopulasi yang berbeda.
PITA_USIA = {
    "diabetes": "15+",
    "hipertensi": "15+",
    "kanker_paru": "45+",
    "tbc": "semua usia",
}

print("Wilayah dibandingkan:", ", ".join(WILAYAH))

## 1. Memuat Hasil Bersih Kedua Wilayah

Yang dibaca adalah `dataset_master.csv` masing-masing wilayah -- keluaran
notebook `Persiapan Data` yang sudah melalui seluruh tahap pembersihan. Notebook
ini **tidak** menyentuh berkas mentah dan tidak mengulang pembersihan apa pun.

In [ ]:
# ===================== MEMUAT MASTER TIAP WILAYAH =====================
master = {}
for w in WILAYAH:
    d = cfg.baca_master(w)
    d["wilayah"] = w
    master[w] = d
    print(f"{w:<8} {len(d):>9,} baris x {d.shape[1]} kolom   "
          f"usia {int(d.Usia.min())}-{int(d.Usia.max())}")

# Skema kedua wilayah harus identik, kalau tidak penggabungan akan diam-diam
# menghasilkan kolom NaN.
kolom = {w: list(d.columns) for w, d in master.items()}
assert kolom["Bantul"] == kolom["Sleman"], "Skema kedua wilayah berbeda!"
print("\nSkema identik:", len(kolom["Bantul"]), "kolom")

## 2. Menyusun Dataset Gabungan

Penggabungan dilakukan dengan `pd.concat` sederhana ditambah satu kolom
`wilayah`. Tidak ada baris yang dibuang di sini: kedua sumber sudah bersih.

`wilayah` bukan sekadar penanda asal. Pada notebook `3. Model Gabungan` kolom ini
menjadi **fitur kategorik ketujuh**, sehingga model gabungan bisa ditanyai
pertanyaan yang tidak bisa dijawab model per wilayah: *setelah usia, jenis
kelamin, segmen dan antropometri diperhitungkan, apakah kabupaten masih
menjelaskan sesuatu?*

In [ ]:
# ===================== GABUNGKAN =====================
gab = pd.concat([master[w] for w in WILAYAH], ignore_index=True)

print(f"Gabungan: {len(gab):,} baris x {gab.shape[1]} kolom")
print()
print(gab["wilayah"].value_counts().rename("baris").to_frame().assign(
    persen=lambda t: (t["baris"] / len(gab) * 100).round(2)))

# Tidak boleh ada nilai hilang setelah penggabungan.
hilang = gab.isna().sum()
print("\nKolom dengan nilai hilang:",
      "tidak ada" if hilang.sum() == 0 else hilang[hilang > 0].to_dict())

In [ ]:
# ===================== DATASET PER TARGET (GABUNGAN) =====================
# Aturannya sama persis dengan notebook Persiapan Data tiap wilayah: baris
# "Tidak Dapat Pertanyaan" dibuang karena orang tersebut memang TIDAK DITANYA,
# lalu label diubah menjadi 0/1.
FITUR = [
    "id", "wilayah", "Jeniskelamin", "Usia", "segmen",
    "Tinggibadan", "Berat Badan", "IMT_recalculated",
    "Kategori_BMI_WHO8", "Kategori_BMI_WHO4",
]

OUT = cfg.folder_data("Gabungan")
OUT.mkdir(parents=True, exist_ok=True)

MASTER_COLS = FITUR[:1] + ["wilayah"] + [
    c for c in master["Bantul"].columns
    if c not in ("id", "wilayah")
]
MASTER_COLS = list(dict.fromkeys(MASTER_COLS))
gab[MASTER_COLS].to_csv(OUT / "dataset_master.csv", index=False,
                        encoding="utf-8-sig")
print(f"dataset_master.csv       -> {len(gab):,} baris x {len(MASTER_COLS)} kolom")

ringkas = []
for nama, kolom_target in TARGETS.items():
    d = gab[FITUR + [kolom_target]].copy()
    d = d[d[kolom_target].isin(cfg.TARGET_MAPPING)]
    d[kolom_target] = d[kolom_target].map(cfg.TARGET_MAPPING).astype(int)
    d.to_csv(OUT / f"dataset_{nama}.csv", index=False, encoding="utf-8-sig")

    ringkas.append({
        "Dataset": nama,
        "Pita usia": PITA_USIA[nama],
        "Baris": f"{len(d):,}",
        "Usia min-maks": f"{int(d.Usia.min())}-{int(d.Usia.max())}",
        "Bantul": f"{int((d.wilayah == 'Bantul').sum()):,}",
        "Sleman": f"{int((d.wilayah == 'Sleman').sum()):,}",
        "% Berisiko": round(d[kolom_target].mean() * 100, 2),
    })
    print(f"dataset_{nama + '.csv':<17} -> {len(d):,} baris")

print()
display(pd.DataFrame(ringkas))
print(f"\nTersimpan di: {OUT}")

Kolom `% Berisiko` di atas adalah prevalensi **gabungan**, dan nilainya selalu
berada di antara kedua wilayah -- rata-rata tertimbang menurut jumlah baris,
bukan rata-rata sederhana. Sleman menyumbang lebih banyak baris pada setiap
dataset, jadi angka gabungan selalu tertarik ke arah Sleman.

In [ ]:
# ===================== VERIFIKASI EKSPOR =====================
# Dibaca ulang dari disk, bukan dari memori: yang diuji adalah berkas yang
# nanti benar-benar dipakai notebook lain dan dashboard.
periksa = []


def cek(nama, lulus, detail=""):
    periksa.append({"Pemeriksaan": nama,
                    "Status": "LULUS" if lulus else "GAGAL",
                    "Detail": detail})


baca = {n: pd.read_csv(OUT / f"dataset_{n}.csv") for n in TARGETS}
baca_master = pd.read_csv(OUT / "dataset_master.csv")

cek("Baris master = jumlah kedua wilayah",
    len(baca_master) == sum(len(master[w]) for w in WILAYAH),
    f"{len(baca_master):,}")

for n, kolom_target in TARGETS.items():
    d = baca[n]
    cek(f"{n}: hanya nilai 0/1",
        set(d[kolom_target].unique()) <= {0, 1}, str(sorted(d[kolom_target].unique())))
    cek(f"{n}: kedua wilayah terwakili",
        set(d["wilayah"]) == set(WILAYAH), str(sorted(set(d["wilayah"]))))
    cek(f"{n}: tidak ada nilai hilang", int(d.isna().sum().sum()) == 0,
        str(int(d.isna().sum().sum())))

    # Prevalensi gabungan harus dapat disusun ulang dari kedua wilayah.
    per_w = d.groupby("wilayah")[kolom_target].agg(["sum", "count"])
    susun = per_w["sum"].sum() / per_w["count"].sum()
    cek(f"{n}: prevalensi tersusun dari dua wilayah",
        abs(susun - d[kolom_target].mean()) < 1e-12, f"{susun:.6f}")

cek("Level segmen sama di semua dataset",
    len({tuple(sorted(d["segmen"].unique())) for d in baca.values()}) == 1,
    str(sorted(baca["tbc"]["segmen"].unique())))

hasil = pd.DataFrame(periksa)
display(hasil)
n_gagal = int((hasil["Status"] == "GAGAL").sum())
print(f"\n{len(hasil) - n_gagal}/{len(hasil)} pemeriksaan lulus.")
assert n_gagal == 0, "Ada pemeriksaan yang gagal -- jangan lanjut."

## 3. Perbandingan Proses Pembersihan

Sebelum membandingkan populasinya, ada baiknya membandingkan **berapa banyak yang
harus dibuang untuk sampai ke sana**. Jejak audit tiap wilayah dibaca dari
`audit_pembersihan.csv`, keluaran notebook Persiapan Data masing-masing.

In [ ]:
# ===================== JEJAK AUDIT KEDUA WILAYAH =====================
audit = {}
for w in WILAYAH:
    a = pd.read_csv(cfg.folder_data(w) / "audit_pembersihan.csv")
    a["Selisih"] = a["Baris tersisa"].diff().fillna(0).astype(int)
    audit[w] = a
    print(f"--- {w} ---")
    display(a)

# Digabung berdasarkan NAMA TAHAP, bukan urutan baris. Kedua wilayah tidak
# dijamin punya daftar tahap yang sama panjang -- tahap "5b" misalnya hanya
# menghasilkan baris terbuang di Sleman. Menggabung berdasarkan posisi akan
# menyandingkan tahap yang berbeda dan memunculkan baris NaN tanpa nama.
banding = audit["Bantul"][["Tahap", "Baris tersisa"]].rename(
    columns={"Baris tersisa": "Bantul"})
for w in WILAYAH[1:]:
    banding = banding.merge(
        audit[w][["Tahap", "Baris tersisa"]].rename(columns={"Baris tersisa": w}),
        on="Tahap", how="outer")

# Urutkan sesuai nomor tahap ("5b" tepat setelah "5").
banding["_urut"] = banding["Tahap"].str.extract(r"^(\d+)")[0].astype(int) * 10 + \
    banding["Tahap"].str.contains(r"^\d+b").astype(int)
banding = banding.sort_values("_urut").drop(columns="_urut").reset_index(drop=True)

for w in WILAYAH:
    banding[f"{w} %sisa"] = (banding[w] / banding[w].dropna().iloc[0] * 100).round(2)

display(banding)

# Tahap yang hanya ada di salah satu wilayah disebutkan, bukan dibiarkan
# muncul sebagai sel kosong yang tidak dijelaskan.
for w in WILAYAH:
    hilang = banding.loc[banding[w].isna(), "Tahap"].tolist()
    if hilang:
        print(f"Tahap yang tidak tercatat pada {w}: {hilang}")
        print("  (jalankan ulang notebook Persiapan Data wilayah itu "
              "supaya jejak auditnya sebanding)")

In [ ]:
# ===================== WATERFALL BERDAMPINGAN =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2, sharey=False)

for ax, w in zip(axes, WILAYAH):
    a = audit[w]
    ax.bar(range(len(a)), a["Baris tersisa"], color=WARNA[w])
    ax.set_xticks(range(len(a)))
    ax.set_xticklabels([t.split(". ", 1)[-1] for t in a["Tahap"]],
                       rotation=18, ha="right", fontsize=8)
    ax.set_title(f"{w} -- {a['Baris tersisa'].iloc[0]:,} menjadi "
                 f"{a['Baris tersisa'].iloc[-1]:,}")
    ax.set_ylabel("Baris tersisa")
    vz.thousands(ax)
    for i, (v, s) in enumerate(zip(a["Baris tersisa"], a["Selisih"])):
        ax.text(i, v, f"{v:,}" + (f"\n({s:+,})" if s else ""),
                ha="center", va="bottom", fontsize=8)
    ax.set_ylim(0, a["Baris tersisa"].max() * 1.18)

fig.suptitle("Tahap Pembersihan Data per Wilayah", fontsize=13)
plt.tight_layout()
plt.show()

Bentuk kurvanya sama di kedua wilayah, dan itu sendiri sudah informatif: jenis
kerusakan datanya sama, bukan masalah khas satu kabupaten. Persentase yang
dibuang berbeda tipis, dan **Bantul membuang proporsi lebih besar** -- terutama
pada tahap batas plausibilitas per kelompok usia, yang menyaring catatan anak
yang tidak mungkin secara fisik.

Satu tahap hanya ada karena Sleman: **`5b. Setelah buang segmen kosong`.** Bantul
tidak punya satu pun baris tanpa `segmen`, Sleman punya satu. Tahapnya tetap
dijalankan di kedua wilayah supaya jejak auditnya sebanding, dan supaya ekspor
berikutnya yang kebetulan banyak baris kosongnya terlihat di angka audit alih-alih
lolos diam-diam sampai tahap pemodelan.

## 4. Profil Populasi

### 4.1 Piramida penduduk peserta skrining

In [ ]:
# ===================== PIRAMIDA USIA x JENIS KELAMIN =====================
AGE_BINS = [0, 5, 15, 25, 35, 45, 55, 65, 120]
AGE_LABELS = ["0-4", "5-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65+"]

gab["Kelompok_Usia"] = pd.cut(gab["Usia"], bins=AGE_BINS, labels=AGE_LABELS,
                              right=False)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2, sharex=True, sharey=True)

for ax, w in zip(axes, WILAYAH):
    d = gab[gab["wilayah"] == w]
    # Sumbu-X dalam PERSEN, bukan jumlah -- Sleman punya lebih banyak peserta,
    # jadi grafik jumlah hanya akan memperlihatkan selisih ukuran sampel.
    tab = (pd.crosstab(d["Kelompok_Usia"], d["Jeniskelamin"])
             .reindex(AGE_LABELS).fillna(0))
    tot = tab.values.sum()
    y = range(len(AGE_LABELS))
    ax.barh(y, -tab["L"] / tot * 100, color=vz.PALETTE["SEX"]["L"], label="Laki-laki")
    ax.barh(y, tab["P"] / tot * 100, color=vz.PALETTE["SEX"]["P"], label="Perempuan")
    ax.set_yticks(list(y))
    ax.set_yticklabels(AGE_LABELS)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"{w}  (n = {len(d):,})")
    ax.set_xlabel("% dari seluruh peserta wilayah")
    ax.grid(axis="x", alpha=0.3)

axes[0].legend(loc="lower right")
axes[0].xaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f"{abs(v):.0f}%"))
fig.suptitle("Piramida Peserta Skrining -- Bantul vs Sleman", fontsize=13)
plt.tight_layout()
plt.show()

display(pd.crosstab(gab["Kelompok_Usia"], gab["wilayah"], normalize="columns")
        .mul(100).round(2).rename_axis("Kelompok usia"))

In [ ]:
# ===================== RINGKASAN DEMOGRAFI =====================
ringkas_demo = []
for w in WILAYAH:
    d = gab[gab["wilayah"] == w]
    ringkas_demo.append({
        "Wilayah": w,
        "Peserta": f"{len(d):,}",
        "% Perempuan": round((d.Jeniskelamin == "P").mean() * 100, 2),
        "Usia rata-rata": round(d.Usia.mean(), 1),
        "Usia median": int(d.Usia.median()),
        "Usia maks": int(d.Usia.max()),
        "% usia 60+": round((d.Usia >= 60).mean() * 100, 2),
        "IMT rata-rata": round(d.IMT_recalculated.mean(), 2),
        "% IMT > 25": round((d.IMT_recalculated > 25).mean() * 100, 2),
    })
display(pd.DataFrame(ringkas_demo).set_index("Wilayah"))

### 4.2 Segmen kepesertaan

`segmen` adalah kategori kepesertaan BPJS, dan pada Bantul ia adalah variabel
dengan pengaruh terbesar pada TBC menurut *gain*. Komposisinya perlu dilihat
lebih dulu, karena **perbedaan komposisi antar wilayah bisa menjelaskan
perbedaan prevalensi tanpa ada perbedaan risiko sama sekali.**

In [ ]:
# ===================== KOMPOSISI SEGMEN =====================
komposisi = (pd.crosstab(gab["segmen"], gab["wilayah"], normalize="columns")
               .mul(100).round(2))
komposisi["Selisih (Sleman-Bantul)"] = (komposisi["Sleman"] - komposisi["Bantul"]).round(2)
komposisi = komposisi.sort_values("Bantul", ascending=False)
display(komposisi)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
y = np.arange(len(komposisi))
tinggi = 0.38
for i, w in enumerate(WILAYAH):
    ax.barh(y + (i - 0.5) * tinggi, komposisi[w], height=tinggi,
            color=WARNA[w], label=w)
ax.set_yticks(y)
ax.set_yticklabels(komposisi.index)
ax.invert_yaxis()
ax.set_xlabel("% peserta di wilayah tersebut")
ax.set_title("Komposisi Segmen Kepesertaan per Wilayah")
ax.legend()
ax.grid(axis="x", alpha=0.3)
for i, w in enumerate(WILAYAH):
    for j, v in enumerate(komposisi[w]):
        ax.text(v + 0.4, j + (i - 0.5) * tinggi, f"{v:.1f}%",
                va="center", fontsize=8)
plt.tight_layout()
plt.show()

Perbedaannya besar dan tidak bisa diabaikan. **`APBN` mendominasi Bantul jauh
lebih kuat daripada Sleman**, sementara Sleman punya proporsi `PBPU` dan
`PPU BU` yang jauh lebih besar. Karena `PBPU` adalah segmen yang pada Bantul
terbukti punya risiko TBC tinggi di *setiap* pita usia, perbedaan komposisi ini
sudah cukup untuk menaikkan prevalensi TBC Sleman tanpa perlu ada perbedaan
risiko per orang.

Notebook `2. Replikasi Temuan` menguji mana dari keduanya yang terjadi.

### 4.3 Status gizi (IMT)

In [ ]:
# ===================== SEBARAN IMT =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# Kiri: kurva kepadatan IMT, dua wilayah pada satu sumbu.
for w in WILAYAH:
    d = gab.loc[gab["wilayah"] == w, "IMT_recalculated"]
    axes[0].hist(d, bins=80, range=(10, 45), density=True, histtype="step",
                 lw=2, color=WARNA[w], label=f"{w} (rata-rata {d.mean():.2f})")
axes[0].axvline(25, color="black", ls="--", lw=1)
axes[0].text(25.2, axes[0].get_ylim()[1] * 0.92, "IMT 25", fontsize=9)
axes[0].set_xlabel("IMT (dihitung ulang)")
axes[0].set_ylabel("Kepadatan")
axes[0].set_title("Sebaran IMT")
axes[0].legend()

# Kanan: kategori WHO-4, hanya untuk DEWASA (>20 th). Anak memakai persentil
# CDC, sehingga labelnya tidak sebanding dengan label dewasa.
dewasa = gab[gab["Usia"] > bs.CHILD_MAX_AGE]
urut = ["Underweight", "Normal", "Overweight", "Obese"]
kat = (pd.crosstab(dewasa["Kategori_BMI_WHO4"], dewasa["wilayah"],
                   normalize="columns").mul(100).reindex(urut).round(2))
x = np.arange(len(urut))
for i, w in enumerate(WILAYAH):
    axes[1].bar(x + (i - 0.5) * 0.38, kat[w], width=0.38, color=WARNA[w], label=w)
    for j, v in enumerate(kat[w]):
        axes[1].text(j + (i - 0.5) * 0.38, v + 0.6, f"{v:.1f}", ha="center", fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(urut)
axes[1].set_ylabel("% peserta dewasa (usia > 20)")
axes[1].set_title("Status Gizi WHO-4, dewasa saja")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
display(kat)

## 5. Prevalensi Tiap Penyakit

Bagian inti perbandingan. Perhatikan bahwa setiap batang punya **pita usianya
sendiri** dan jumlah barisnya berbeda-beda -- keempat penyakit tidak ditanyakan
pada populasi yang sama.

In [ ]:
# ===================== PREVALENSI PER PENYAKIT =====================
prev = []
for nama, kolom_target in TARGETS.items():
    d = baca[nama]
    baris = {"Penyakit": NAMA_PENYAKIT[nama], "Pita usia": PITA_USIA[nama]}
    for w in WILAYAH:
        dw = d[d["wilayah"] == w]
        baris[f"n {w}"] = len(dw)
        baris[f"% {w}"] = round(dw[kolom_target].mean() * 100, 2)
    baris["Selisih (poin %)"] = round(baris["% Sleman"] - baris["% Bantul"], 2)
    baris["Rasio"] = round(baris["% Sleman"] / baris["% Bantul"], 3)
    prev.append(baris)

tabel_prev = pd.DataFrame(prev)
display(tabel_prev)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
x = np.arange(len(tabel_prev))
for i, w in enumerate(WILAYAH):
    v = tabel_prev[f"% {w}"]
    ax.bar(x + (i - 0.5) * 0.38, v, width=0.38, color=WARNA[w], label=w)
    for j, val in enumerate(v):
        ax.text(j + (i - 0.5) * 0.38, val + 0.7, f"{val:.2f}%", ha="center",
                fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f"{p}\n({u})" for p, u in
                    zip(tabel_prev["Penyakit"], tabel_prev["Pita usia"])])
ax.set_ylabel("% Berisiko")
ax.set_title("Prevalensi Risiko per Penyakit -- pita usia dicantumkan di sumbu")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, tabel_prev[[f"% {w}" for w in WILAYAH]].values.max() * 1.15)
plt.tight_layout()
plt.show()

Tiga dari empat penyakit lebih tinggi di Sleman, satu lebih rendah. Yang penting
dibaca bukan arahnya saja tetapi **ukurannya**: selisih beberapa poin persen pada
Diabetes dan Hipertensi berbeda sifatnya dari selisih pada TBC, karena basis
prevalensinya berbeda jauh. Kolom `Rasio` ada untuk itu.

Ingat juga bahwa **Diabetes bukan prediksi penyakit**. Label Diabetes pada data
ini praktis merupakan aturan `IMT > 25` yang dijalankan kuesioner; perbedaan
prevalensi Diabetes antar wilayah karenanya sebagian besar hanyalah perbedaan
proporsi peserta ber-IMT di atas 25. Bukti angkanya ada di notebook
`2. Replikasi Temuan`.

In [ ]:
# ===================== PREVALENSI PER KELOMPOK USIA =====================
# Yang digambar PREVALENSI, bukan jumlah orang berisiko: jumlah orang berisiko
# adalah "peserta x risiko" dan suku pertamanya biasanya mendominasi, sehingga
# grafik jumlah bisa membalik kesimpulan.
fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, (nama, kolom_target) in zip(axes.ravel(), TARGETS.items()):
    d = baca[nama].copy()
    d["Kelompok_Usia"] = pd.cut(d["Usia"], bins=AGE_BINS, labels=AGE_LABELS,
                                right=False)
    for w in WILAYAH:
        dw = d[d["wilayah"] == w]
        g = dw.groupby("Kelompok_Usia", observed=True)[kolom_target].agg(
            ["mean", "size"])
        g = g[g["size"] >= 50]          # pita terlalu kecil tidak digambar
        ax.plot([AGE_LABELS.index(str(i)) for i in g.index], g["mean"] * 100,
                marker="o", color=WARNA[w], label=w)
    ax.set_xticks(range(len(AGE_LABELS)))
    ax.set_xticklabels(AGE_LABELS, rotation=30)
    ax.set_title(f"{NAMA_PENYAKIT[nama]}  (ditanyakan: {PITA_USIA[nama]})")
    ax.set_ylabel("% Berisiko")
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle("Prevalensi menurut Kelompok Usia -- pita dengan n < 50 tidak digambar",
             fontsize=13)
plt.tight_layout()
plt.show()

Kurvanya sejajar, dan lebih rapi daripada yang biasanya terjadi. Diperiksa pita
per pita (hanya pita dengan n >= 50 di kedua wilayah):

| Penyakit | Pita dibandingkan | Sleman lebih tinggi | Selisih |
|---|---|---|---|
| Diabetes Melitus | 6 | **6 dari 6** | +1,46 sampai +5,23 poin |
| Hipertensi | 6 | **6 dari 6** | +1,05 sampai +3,11 poin |
| TBC | 8 | **8 dari 8** | +0,41 sampai +1,09 poin |
| Kanker Paru | 3 | **0 dari 3** | -0,51 sampai -1,11 poin |

**23 dari 23 perbandingan searah dengan selisih totalnya.** Arahnya tidak
berubah di satu pita pun, jadi selisih antar wilayah adalah **pergeseran
tingkat**, bukan pengaruh yang hanya muncul pada kelompok usia tertentu.

Satu perbedaan bentuk yang nyata dan perlu disebut: pada TBC, prevalensi Bantul
**turun** setelah 55-64 (11,08% menjadi 10,70%), sedangkan Sleman terus **naik**
(11,70% menjadi 11,77%). Selisihnya kecil, tetapi arahnya berlawanan.

Perhatikan pula bahwa kurva Kanker Paru hanya dimulai dari pita `45-54`, dan
kurva Diabetes/Hipertensi dari `15-24`. Bagian kiri yang kosong bukan data
hilang, melainkan kelompok usia yang memang tidak pernah ditanyai.

## 6. Ringkasan

| Yang dibandingkan | Temuan |
|---|---|
| Jumlah peserta | Sleman lebih besar; keduanya ratusan ribu catatan skrining |
| Proses pembersihan | Bentuk waterfall sama, proporsi dibuang beda tipis; hanya Sleman punya baris tanpa `segmen` |
| Sebaran usia & jenis kelamin | Serupa, sama-sama didominasi perempuan |
| Segmen kepesertaan | **Berbeda nyata** -- `APBN` jauh lebih dominan di Bantul, `PBPU` dan `PPU BU` jauh lebih besar di Sleman |
| Prevalensi | Sleman lebih tinggi pada tiga penyakit, lebih rendah pada Kanker Paru |
| Bentuk hubungan dengan usia | Sama di kedua wilayah |

Dua pertanyaan yang belum terjawab di sini dan dijawab notebook berikutnya:

1. **Apakah temuan-temuan Bantul berlaku juga di Sleman?** Aturan `IMT > 25` pada
   Diabetes, tinggi badan yang sebenarnya menandai jenis kelamin pada Kanker
   Paru, dan segmen mana yang sebenarnya mendorong TBC. -> `2. Replikasi Temuan`
2. **Apakah selisih prevalensi TBC itu nyata, atau hanya akibat komposisi segmen
   yang berbeda?** -> `2. Replikasi Temuan` dan `3. Model Gabungan`

Berkas yang dihasilkan notebook ini -- `dataset_master.csv` dan empat
`dataset_*.csv` di `Gabungan/Data Setiap Target/` -- menjadi masukan notebook
`3. Model Gabungan`.